# Behavioral Model

This notebook demonstrates the LimSim-style interactive behavior model in Tactics2D.

The implementation follows the original LimSim PDP structure at a Tactics2D-native level: local interaction grouping, MCT/MCTS-based joint behavior decisions, and a Frenet-style final trajectory planner on lane geometry. It outputs normal Tactics2D `Trajectory` objects, so the result can be evaluated, visualized, or reused by downstream modules.

The current planner is still lighter than LimSim's full official planner stack, but it now includes reference-path Frenet conversion, quartic/quintic polynomial trajectory sampling, lane-change candidates, and dynamic-obstacle costs.


## 1. Import dependencies

The imports follow the same style as the other tutorial notebooks. If direct imports fail in a fresh local environment, install the repository in editable mode first with `pip install -e .`.

In [ ]:
%matplotlib inline

import warnings
from math import hypot
from pathlib import Path
import sys

warnings.filterwarnings("ignore")

def find_repo_root(start):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "tactics2d" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Cannot find the Tactics2D repository root.")

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
from shapely.geometry import LineString

from tactics2d.behavior import LimSimBehaviorModel
from tactics2d.behavior.limsim.config import LimSimConfig
from tactics2d.behavior.limsim.evaluation import (
    dimensions_from_participants,
    evaluate_planning_result,
)
from tactics2d.behavior.limsim.frenet_planner import (
    FrenetTrajectoryPlanner,
    ReferencePath,
)
from tactics2d.behavior.limsim.prediction import LimSimPredictor
from tactics2d.behavior.limsim.roi import RoISelector
from tactics2d.map.element import Lane, LaneRelationship, Map
from tactics2d.participant.element import Vehicle
from tactics2d.participant.trajectory import State, Trajectory


In [ ]:
def build_lane(lane_id, x_left, x_right, y_start, y_end):
    left_side = LineString([(x_left, y_start), (x_left, y_end)])
    right_side = LineString([(x_right, y_start), (x_right, y_end)])
    return Lane(
        id_=lane_id,
        left_side=left_side,
        right_side=right_side,
        custom_tags={
            "centerline": np.array(
                [[(x_left + x_right) / 2.0, y_start], [(x_left + x_right) / 2.0, y_end]]
            )
        },
    )


def build_vehicle(agent_id, frame, x, y, heading=np.pi / 2, speed=5.0):
    trajectory = Trajectory(id_=agent_id, fps=10, stable_freq=True)
    trajectory.add_state(
        State(
            frame=frame,
            x=x,
            y=y,
            heading=heading,
            vx=speed * np.cos(heading),
            vy=speed * np.sin(heading),
        )
    )
    return Vehicle(agent_id, "vehicle", trajectory=trajectory, length=4.5, width=1.8)

## 2. What this model currently does

At one simulation frame, the model performs the following steps:

1. Convert Tactics2D participants and map lanes into compact decision states.
2. Build local interaction groups using distance and lane-topology rules.
3. Run group-wise MCTS over LimSim-style actions: `KS`, `AC`, `DC`, `LCL`, and `LCR`.
4. Convert the selected action into a final trajectory with a Tactics2D Frenet-style planner.
5. Optionally evaluate action distribution, predicted collisions, and ADE/FDE against dataset ground truth.

The final planner is separate from the MCTS rollout model. MCTS uses a fast lane-following rollout to search action combinations, then the final planner samples polynomial Frenet trajectories and scores them against background and group obstacles.


In [ ]:
map_ = Map(name="limsim_tutorial_map")
lane_a = build_lane("A", 0.0, 2.0, 0.0, 80.0)
lane_b = build_lane("B", 2.0, 4.0, 0.0, 80.0)
lane_a.add_related_lane("B", LaneRelationship.RIGHT_NEIGHBOR)
lane_b.add_related_lane("A", LaneRelationship.LEFT_NEIGHBOR)
map_.add_lane(lane_a)
map_.add_lane(lane_b)

participants = {
    1: build_vehicle(1, 0, 1.0, 5.0, speed=5.0),
    2: build_vehicle(2, 0, 1.0, 11.0, speed=2.0),
    3: build_vehicle(3, 0, 3.0, 45.0, speed=4.0),
}

## 3. Run LimSim-style MCT planning

The returned result contains interaction groups, selected high-level actions, MCTS root nodes, and planned Tactics2D trajectories. In this small scene, vehicles 1 and 2 are close enough to be planned as one interaction group, while vehicle 3 is far away and can be planned independently.

MCTS uses a lightweight rollout internally for action search. The final `result.trajectories` are generated by the Frenet-style planner.


In [ ]:
config = LimSimConfig(horizon_steps=20, mcts_iterations=60, interaction_distance=20.0)
behavior_model = LimSimBehaviorModel(config)
result = behavior_model.plan(participants, map_, frame=0)
evaluation = evaluate_planning_result(
    result,
    dimensions=dimensions_from_participants(participants),
)

print("interaction groups:", result.groups)
print("selected actions:", {agent_id: action.value for agent_id, action in result.actions.items()})
print("evaluation:", {"actions": evaluation.action_counts, "collision": evaluation.has_collision})

scene_states = behavior_model.scene_builder.build(participants, map_, frame=0)
reference_path = ReferencePath.from_agent(scene_states[1], map_, config)
sampled = FrenetTrajectoryPlanner(config).sample_candidates(
    scene_states[1], result.actions[1], reference_path, map_
)
print("frenet candidates for agent 1:", len(sampled))
print("best candidate cost:", round(min(candidate.cost for candidate in sampled), 3))


## 4. Inspect planned and predicted trajectories

`result.trajectories` contains the final Frenet-style planned trajectories after MCTS selects actions. `LimSimPredictor` provides the lightweight prediction step used by the reproduced pipeline: if a previous plan is available it can reuse the remaining planned trajectory; otherwise it rolls the participant forward with the current lane-following rule.


In [ ]:
predictor = LimSimPredictor(config)
prediction = predictor.predict(participants, map_, frame=0)

for agent_id, trajectory in result.trajectories.items():
    trace = trajectory.get_trace()
    print("planned", agent_id, result.actions[agent_id].value, trace[:3], "...", trace[-3:])

for agent_id, trajectory in prediction.items():
    print("predicted", agent_id, len(trajectory.frames), "states")


## 5. WOMD RoI examples

For WOMD, use the existing parser and pass the parsed participants and map directly into the same behavior model. The examples below use a small Region of Interest (RoI): vehicles inside the RoI are controlled by PDP, while vehicles in the outer band are kept as predicted background obstacles for the final planner.

Two representative interaction cases are shown from the local interactive validation sample:

- **ego-centered RoI**: center follows ego vehicle `4098`, radius `8 m`. Nearby vehicles are close enough that PDP selects several deceleration actions.
- **fixed physical RoI**: center is fixed near `(-854.1, -686.4)`, radius `10 m`. Three close vehicles are selected and all decelerate.

Both examples keep the radius small so the tutorial remains fast and the local interaction is easy to inspect. The output is textual on purpose: the printed RoI agents, background agents, interaction groups, actions, collision flag, and closest planned distance are the key signals for checking whether the interaction logic is behaving reasonably.


In [ ]:
from tactics2d.dataset_parser import WOMDParser


def min_planned_distance(planning_result):
    """Return the closest pair distance among planned trajectories."""

    closest = None
    items = list(planning_result.trajectories.items())
    for index, (agent_id, trajectory) in enumerate(items):
        for other_id, other_trajectory in items[index + 1 :]:
            shared_frames = sorted(set(trajectory.frames).intersection(other_trajectory.frames))
            for frame in shared_frames:
                state = trajectory.get_state(frame)
                other_state = other_trajectory.get_state(frame)
                distance = hypot(state.x - other_state.x, state.y - other_state.y)
                if closest is None or distance < closest["distance"]:
                    closest = {
                        "distance": distance,
                        "agents": (agent_id, other_id),
                        "frame": frame,
                    }
    return closest


def run_womd_case(case, participants, map_, config):
    behavior_model = LimSimBehaviorModel(config)
    if case["mode"] == "ego":
        planning_result = behavior_model.plan(
            participants,
            map_,
            case["frame"],
            ego_id=case["ego_id"],
            roi_radius=case["roi_radius"],
        )
    else:
        planning_result = behavior_model.plan(
            participants,
            map_,
            case["frame"],
            roi_center=case["roi_center"],
            roi_radius=case["roi_radius"],
        )

    selected = {agent_id: participants[agent_id] for agent_id in planning_result.roi_agent_ids}
    evaluation = evaluate_planning_result(
        planning_result,
        dimensions=dimensions_from_participants(selected),
    )
    closest = min_planned_distance(planning_result)
    return planning_result, evaluation, closest


folder = repo_root / "tactics2d" / "data" / "trajectory_sample" / "WOMD"
file_name = "uncompressed_scenario_validation_interactive_validation_interactive.tfrecord-00000-of-00150"
scenario_index = 1

cases = [
    {
        "name": "ego-centered RoI",
        "mode": "ego",
        "frame": 0,
        "ego_id": 4098,
        "roi_radius": 8.0,
    },
    {
        "name": "fixed physical RoI",
        "mode": "fixed",
        "frame": 1000,
        "roi_center": (-854.1, -686.4),
        "roi_radius": 10.0,
    },
]

if (folder / file_name).exists():
    parser = WOMDParser()
    participants_womd, _ = parser.parse_trajectory(
        scenario_index, file=file_name, folder=str(folder)
    )
    map_womd = parser.parse_map(scenario_index, file=file_name, folder=str(folder))
    womd_config = LimSimConfig(horizon_steps=8, mcts_iterations=15, interaction_distance=16.0)

    for case in cases:
        planning_result, evaluation, closest = run_womd_case(
            case, participants_womd, map_womd, womd_config
        )
        print("\n", case["name"])
        print("RoI agents:", planning_result.roi_agent_ids)
        print("background agents:", planning_result.background_agent_ids)
        print("groups:", planning_result.groups)
        print("actions:", {agent_id: action.value for agent_id, action in planning_result.actions.items()})
        print("collision:", evaluation.has_collision)
        if closest is not None:
            print(
                "closest planned distance:",
                round(closest["distance"], 3),
                "between",
                closest["agents"],
                "at frame",
                closest["frame"],
            )
else:
    print("WOMD sample file is not available in this checkout:", folder / file_name)


## 6. Current scope and limitations

This reproduction currently supports RoI selection, interaction grouping, MCTS behavior selection, lane-topology-aware rollout, Frenet-style final trajectory sampling, dynamic background-obstacle scoring, rule-based prediction, and basic evaluation. It can be used to inspect whether agents are grouped reasonably, whether selected actions are plausible, and whether short-horizon planned trajectories collide.

Compared with the official LimSim planner, this is still an intermediate Tactics2D-native version. It now uses planner-facing map semantic queries for lane width, speed limit, lane-change permission, stop targets, traffic-light state, and junction conflict checks, but it does not yet reproduce the exact official full planner and cost stack. The current layer is suitable for validating the PDP interaction pipeline and can be extended with richer stop/yield/traffic-light/junction trajectory rules as map semantics mature.
